In [ ]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import savgol_filter
from scipy.stats import linregress


In [ ]:
spots_all=  pd.read_csv("concatenated_trackmate_spots.csv")

In [ ]:
spots = spots_all.copy(deep=True)

In [ ]:
pixel_to_micron = 1.5 #pixels per micron
frame_rate = 0.2  #fps

In [ ]:


import seaborn as sns
sns.set_theme(style="ticks", font="Helvetica", font_scale=1.4)

# ------------------ USER PARAMS ---------------------------------------------------------
min_distance_um  = 500.0    # set None to disable distance filter or input number in microns ; filters out trajectories smaller than defined bumber
use_savgol       = True     # smooth positions before MSD (AFTER filtering) # Savitsky-Golay filter
SG_WINDOW        = 9        # odd, >=3 ; will auto-shrink for short tracks
SG_POLY          = 2
max_lag_fraction = 0.5      # compute MSD up to this fraction of each track length (0.1-0.5 is typical)

MAX_GAPS_PER_TRACK = 2      # allow up to this many frame gaps per track
# ----------------------------------------------------------------------------------------

spots = spots_all.copy(deep=True)

needed = {"TRACK_ID", "FRAME", "POSITION_X", "POSITION_Y"}
missing = needed - set(spots.columns)
if missing:
    raise ValueError(f"CSV missing required columns: {missing}")

spots = spots[["TRACK_ID","FRAME","POSITION_X","POSITION_Y"]].copy()
spots = spots.sort_values(["TRACK_ID","FRAME"]).reset_index(drop=True)

# --- Gap filter -------------------------------------------------------------------------
def _num_frame_gaps(g):
    fr = g["FRAME"].to_numpy()
    return int(np.sum(np.diff(fr) != 1))

gap_counts = spots.groupby("TRACK_ID", sort=False).apply(_num_frame_gaps)
keep_ids_gap = gap_counts[gap_counts <= MAX_GAPS_PER_TRACK].index
spots = spots[spots["TRACK_ID"].isin(keep_ids_gap)].copy()
print(f"Tracks kept after allowing ≤{MAX_GAPS_PER_TRACK} frame gaps: {len(keep_ids_gap)}")

# --- Distance filter computed on RAW positions ------------------------------------------
def _total_distance_um_raw(g):
    dx = g["POSITION_X"].diff()
    dy = g["POSITION_Y"].diff()
    step_px = np.sqrt(dx**2 + dy**2)
    return float(step_px.fillna(0).sum() * pixel_to_micron)

if min_distance_um is not None:
    keep_ids = [tid for tid, g in spots.groupby("TRACK_ID") if _total_distance_um_raw(g) >= float(min_distance_um)]
    spots = spots[spots["TRACK_ID"].isin(keep_ids)].copy()
    print(f"Tracks kept after raw distance ≥ {min_distance_um:.1f} µm: {len(keep_ids)}")

# --- Optional: Savitzky–Golay smoothing now (AFTER filtering) ---------------------------
def _odd_leq(n):
    k = int(n)
    if k < 3: return 3
    return k if k % 2 else k-1

if use_savgol and not spots.empty:
    def _smooth_group(g):
        n = len(g)
        w = _odd_leq(min(SG_WINDOW, n))
        p = min(SG_POLY, w-1)
        if n >= 3:
            g = g.copy()
            g["POSITION_X"] = savgol_filter(g["POSITION_X"].to_numpy(), w, p, mode="interp")
            g["POSITION_Y"] = savgol_filter(g["POSITION_Y"].to_numpy(), w, p, mode="interp")
        return g
    spots = spots.groupby("TRACK_ID", group_keys=False).apply(_smooth_group)

# --- MSD calculation (per track, then ensemble) -----------------------------------------
def msd_for_track(frames, x, y, max_lag_frames=None):
    frames = frames.to_numpy().astype(int)
    x = x.to_numpy().astype(float)
    y = y.to_numpy().astype(float)

    n = len(frames)
    if n < 2:
        return pd.Series(dtype=float)

    max_lag = max_lag_frames if max_lag_frames is not None else int(np.floor(n * max_lag_fraction))
    max_lag = max(1, min(max_lag, n-1))

    msd_vals = []
    lags = np.arange(1, max_lag+1, dtype=int)
    for lag in lags:
        i0 = np.arange(n - lag)
        i1 = np.arange(lag, n)
        mask = (frames[i1] - frames[i0]) == lag
        if not np.any(mask):
            msd_vals.append(np.nan)
            continue
        dx = x[i1[mask]] - x[i0[mask]]
        dy = y[i1[mask]] - y[i0[mask]]
        msd_px2 = np.mean(dx*dx + dy*dy)
        msd_vals.append(msd_px2 * (pixel_to_micron**2))  # µm^2

    t_sec = lags / float(frame_rate)
    return pd.Series(msd_vals, index=t_sec, dtype=float)

track_msd = {}
for tid, g in spots.groupby("TRACK_ID"):
    s = msd_for_track(g["FRAME"], g["POSITION_X"], g["POSITION_Y"])
    if len(s) > 0:
        track_msd[tid] = s.dropna()

if not track_msd:
    raise ValueError("No valid tracks for MSD (after filtering and/or smoothing).")

imsd_df = pd.DataFrame(track_msd).sort_index()
emsd = imsd_df.mean(axis=1)

# --- Alpha fit with CI  --------------------------------------------------
t = emsd.index.to_numpy(float)
y = emsd.to_numpy(float)
mask = np.isfinite(t) & np.isfinite(y) & (t > 0) & (y > 0)

if mask.sum() >= 3:
    log_t = np.log10(t[mask])
    log_y = np.log10(y[mask])
    slope, intercept, r_value, p_value, std_err = linregress(log_t, log_y)
    alpha = float(slope)
    ci95 = float(1.96 * std_err)  # ~95% CI half-width

    tt = np.logspace(np.log10(t[mask].min()), np.log10(t[mask].max()), 200)
    yy = 10**intercept * tt**alpha
else:
    alpha, ci95, tt, yy = np.nan, np.nan, None, None

# --- Single plot: individual IMSDs + dark gray ensemble + red fit -----------------------
plt.figure(figsize=(5, 4))

for tid in imsd_df.columns:
    plt.plot(imsd_df.index, imsd_df[tid], color='dimgray', alpha=0.12, linewidth=1)

# dark green ensemble line
plt.plot(
    emsd.index,
    emsd.values,
    color='olive',
    linewidth=6,
    solid_capstyle='round',   # smooth line ends
    solid_joinstyle='bevel',  # bevel bends
    label='Ensemble MSD'
)



plt.xscale('log'); plt.yscale('log')
plt.xlabel('Lag time t (s)')
plt.ylabel(r'$\langle \Delta r^2 \rangle$  (µm$^2$)')
plt.title('Individual MSDs + Ensemble MSD')
# plt.legend()
plt.tight_layout()
# plt.savefig("MSD_final2.png", dpi=300, bbox_inches='tight')
plt.show()

def classify_alpha(a, tol=0.1):
    if not np.isfinite(a): return "unknown"
    if a < 1 - tol: return "subdiffusive"
    if a > 1 + tol: return "superdiffusive"
    return "diffusive"

print(f"Tracks used: {imsd_df.shape[1]} | lags used (rows): {imsd_df.shape[0]}")
print("Alpha (ensemble MSD):", f"{alpha:.3f} ± {ci95:.3f}" if np.isfinite(alpha) else "NaN")
print("Motion classification:", classify_alpha(alpha))
# ========================================================================================

In [ ]:
# ── Per-particle alpha SD ─────────────────────────────────────────
# Run this after Code 1 — imsd_df already exists
individual_slopes = []

for col in imsd_df.columns:
    series = imsd_df[col].dropna()
    if len(series) < 3:
        continue
    log_x = np.log10(series.index.to_numpy())
    log_y = np.log10(series.values)
    if not (np.all(np.isfinite(log_x)) and np.all(np.isfinite(log_y))):
        continue
    slope = np.polyfit(log_x, log_y, 1)[0]
    individual_slopes.append(slope)

mean_alpha = np.mean(individual_slopes)
sd_alpha   = np.std(individual_slopes)
n          = len(individual_slopes)

print(f"Particles used : {n}")
print(f"α = {mean_alpha:.4f} ± {sd_alpha:.4f}  (mean ± SD across {n} particles)")
print(f"α (ensemble fit, CI95) = {alpha:.3f} ± {ci95:.3f}")  # Code 1's original value